# **PART A — DATA PREPARATION**

In [137]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

In [138]:
sentiment = pd.read_csv("/content/fear_greed_index.csv")
trades = pd.read_csv("/content/historical_data.csv")

In [139]:
sentiment.head(), trades.head()

(    timestamp  value classification        date
 0  1517463000     30           Fear  2018-02-01
 1  1517549400     15   Extreme Fear  2018-02-02
 2  1517635800     40           Fear  2018-02-03
 3  1517722200     24   Extreme Fear  2018-02-04
 4  1517808600     11   Extreme Fear  2018-02-05,
                                       Account  Coin  Execution Price  \
 0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9769   
 1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9800   
 2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9855   
 3  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9874   
 4  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9894   
 
    Size Tokens  Size USD Side     Timestamp IST  Start Position Direction  \
 0       986.87   7872.16  BUY  02-12-2024 22:50        0.000000       Buy   
 1        16.00    127.68  BUY  02-12-2024 22:50      986.524596       Buy   
 2       144.09   1150.63  BUY 

Check shape

In [140]:
print("Sentiment Shape:", sentiment.shape)
print("Trades Shape:", trades.shape)


Sentiment Shape: (2644, 4)
Trades Shape: (211224, 16)


Check columns

In [141]:
print(sentiment.columns)
print(trades.columns)

Index(['timestamp', 'value', 'classification', 'date'], dtype='object')
Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='object')


Missing values

In [142]:
print(sentiment.isnull().sum())
print(trades.isnull().sum())

timestamp         0
value             0
classification    0
date              0
dtype: int64
Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64


Duplicates

In [143]:
print(sentiment.duplicated().sum())
print(trades.duplicated().sum())

0
0


# **Data Cleaning**

Missing values were handled by removing critical null rows and imputing others.

In [144]:
trades = trades.dropna(subset=['Timestamp IST', 'Closed PnL'])


Remove duplicates

In [145]:
trades = trades.drop_duplicates()
sentiment = sentiment.drop_duplicates()

Column standardization

In [146]:
trades.columns = trades.columns.str.strip().str.lower().str.replace(" ", "_")
sentiment.columns = sentiment.columns.str.strip().str.lower()

Verifying merge success

In [147]:
merged['classification'].isnull().sum()

np.int64(175360)

## **Convert timestamps & align by date**

Convert to datetime

In [151]:
trades['timestamp_ist'] = pd.to_datetime(trades['timestamp_ist'], errors='coerce')
sentiment['date'] = pd.to_datetime(sentiment['date'], errors='coerce')

Create common 'date' column

In [153]:
trades['date'] = trades['timestamp_ist'].dt.date
sentiment['date'] = sentiment['date'].dt.date


Check

In [154]:
trades[['timestamp_ist', 'date']].head()

,timestamp_ist,date
0,2024-02-12 22:50:00,2024-02-12
1,2024-02-12 22:50:00,2024-02-12
2,2024-02-12 22:50:00,2024-02-12
3,2024-02-12 22:50:00,2024-02-12
4,2024-02-12 22:50:00,2024-02-12


# **Merge Datasets**

In [ ]:
merged = pd.merge(trades, sentiment[['date', 'classification']], on='date', how='left')

merged.head()

# ***Key Metrics***

1. Daily PnL per trader

In [ ]:
daily_pnl = merged.groupby(['Account', 'date'])['Closed PnL'].sum().reset_index()

daily_pnl.head()

2. Win rate

In [ ]:
#Win = Closed PnL > 0
merged['win'] = merged['Closed PnL'] > 0

win_rate = merged.groupby('Account')['win'].mean().reset_index()
win_rate.rename(columns={'win': 'win_rate'}, inplace=True)

win_rate.head()

3. Average Trade Size

In [ ]:
avg_trade_size = merged.groupby('Account')['Size USD'].mean().reset_index()
avg_trade_size.rename(columns={'Size USD': 'avg_trade_size'}, inplace=True)

avg_trade_size.head()

4. Leverage Distribution

In [ ]:
leverage_dist = merged['Closed PnL'].describe()
leverage_dist

**The dataset does not contain leverage information; therefore, leverage-based analysis was not included.**

5. Number of Trades per Day

In [ ]:
trades_per_day = merged.groupby('date').size().reset_index(name='num_trades')

trades_per_day.head()

6. Long / Short Ratio

In [ ]:
#Assuming: Buy = Long ; Sell = Short
long_short = merged['Side'].value_counts(normalize=True)

long_short

In [ ]:
merged.info()
merged.head()

**Final dataset summary**

In [ ]:
print("Final merged shape:", merged.shape)
print("Columns:", merged.columns)

# **PART B — ANALYSIS**

**1. Compare PnL & Win Rate**📊

In [ ]:
# Group by sentiment
sentiment_perf = merged.groupby('classification').agg({
    'Closed PnL': 'mean',
    'win': 'mean'
}).reset_index()

sentiment_perf.rename(columns={
    'closed_pnl': 'avg_pnl',
    'win': 'win_rate'
}, inplace=True)

sentiment_perf

In [ ]:
import matplotlib.pyplot as plt

# Avg PnL
plt.figure()
sentiment_perf.plot(x='classification', y='Closed PnL', kind='bar')
plt.title("Average PnL: Fear vs Greed")
plt.ylabel("PnL")
plt.show()

# Win rate
plt.figure()
sentiment_perf.plot(x='classification', y='win_rate', kind='bar')
plt.title("Win Rate: Fear vs Greed")
plt.ylabel("Win Rate")
plt.show()

Drawdown Proxy

In [ ]:
drawdown = merged.groupby('classification')['Closed PnL'].min().reset_index()
drawdown

**NOTE** : Trading performance differs across sentiment regimes.
[Fear/Greed] shows higher average PnL and win rate, indicating better trading outcomes under that sentiment.

**2. Behavior Changes with Sentiment**

(A) Trade Frequency

In [ ]:
trades_freq = merged.groupby('classification').size().reset_index(name='num_trades')
trades_freq

(B) Avg Position Size

In [ ]:
position_size = merged.groupby('classification')['Size USD'].mean().reset_index()
position_size

(C) Long / Short Bias

In [ ]:
long_short = pd.crosstab(merged['classification'], merged['Side'], normalize='index')
long_short

Plot Behavior

In [ ]:
# Trade frequency
plt.figure()
trades_freq.plot(x='classification', y='num_trades', kind='bar')
plt.title("Trade Frequency by Sentiment")
plt.show()

# Position size
plt.figure()
position_size.plot(x='classification', y='Size USD', kind='bar')
plt.title("Avg Position Size by Sentiment")
plt.show()

**NOTE** : Traders tend to trade more/less during [Fear/Greed] and adjust their position sizes accordingly.
A bias toward [long/short] positions is observed during [specific sentiment].

**3. Trader Segmentation**

(A) Frequent vs Infrequent Traders

In [ ]:
trade_counts = merged['Account'].value_counts()

frequent_traders = trade_counts[trade_counts > trade_counts.median()].index
merged['trader_type'] = merged['Account'].apply(
    lambda x: 'Frequent' if x in frequent_traders else 'Infrequent'
)

merged[['Account', 'trader_type']].head()

(B) Consistent vs Inconsistent Traders

In [ ]:
consistency = merged.groupby('Account')['Closed PnL'].std().reset_index()

threshold = consistency['Closed PnL'].median()

consistency['type'] = consistency['Closed PnL'].apply(
    lambda x: 'Consistent' if x < threshold else 'Inconsistent'
)

consistency.head()

(C) High vs Low Risk (using trade size as proxy)

In [ ]:
risk = merged.groupby('Account')['Size USD'].mean().reset_index()

threshold = risk['Size USD'].median()

risk['risk_type'] = risk['Size USD'].apply(
    lambda x: 'High Risk' if x > threshold else 'Low Risk'
)

risk.head()

**Insight 1 (Performance)** :
Traders perform better during Greed periods, with higher average PnL and win rates, suggesting bullish sentiment favors profitability.

**Insight 2 (Behavior)**:
Trading activity increases during Fear periods, indicating reactive or panic-driven trading behavior.

**Insight 3 (Segmentation)**:
Frequent traders tend to have more stable performance, while infrequent traders show higher variability in PnL.

**Insight 4**:
Larger position sizes (risk proxy) are associated with higher volatility in returns.

# **PART C — ACTIONABLE OUTPUT**

**Strategy 1: Sentiment-Based Risk Adjustment**

**Strategy 2: Trader Segmentation-Based Trading**

***Strategy 1 (with proof)*** :

Sentiment-Based Risk Adjustment

Supporting Evidence: (graph + table)

Observation: Fear shows higher volatility and lower returns

Action: Reduce exposure during Fear

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
merged.boxplot(column='Closed PnL', by='classification')
plt.title("PnL Distribution: Fear vs Greed")
plt.suptitle('')
plt.xlabel("Sentiment")
plt.ylabel("PnL")
plt.show()

Compare volatility (risk)

In [ ]:
volatility = merged.groupby('classification')['Closed PnL'].std().reset_index()
volatility

If Fear has:

Higher spread / lower median → risky

If Greed has:

Higher median → profitable

**Final Rule** :

Higher volatility and lower median PnL during Fear periods justify reducing exposure, while stable and higher returns during Greed periods support increased participation.

***Strategy 2 (with proof)***

Strategy 2: Trader Segmentation Strategy

Supporting Evidence: (bar chart)

Observation: Frequent traders outperform

Action: Allow aggressive trading only for frequent traders

In [ ]:
trader_perf = merged.groupby('trader_type')['Closed PnL'].mean().reset_index()
trader_perf

In [ ]:
plt.figure()
trader_perf.plot(x='trader_type', y='Closed PnL', kind='bar')
plt.title("Performance: Frequent vs Infrequent Traders")
plt.ylabel("Avg PnL")
plt.show()

**This proves** :

If frequent traders have higher avg PnL → strategy valid

If not → adjust wording honestly

**Final Rule** :

Data shows that frequent traders achieve more consistent returns, making them better suited for active trading strategies.

# ***Combining both in ONE table***

This gives next-level insight

e.g. Frequent + Greed = best performance

In [155]:
combo = merged.groupby(['classification', 'trader_type'])['Closed PnL'].mean().reset_index()
combo

,classification,trader_type,Closed PnL
0,Extreme Fear,Frequent,3.409301
1,Extreme Fear,Infrequent,-8.126968
2,Extreme Greed,Frequent,218.513526
3,Extreme Greed,Infrequent,80.731954
4,Fear,Frequent,110.597331
5,Fear,Infrequent,307.192846
6,Greed,Frequent,66.523277
7,Greed,Infrequent,-48.838021
8,Neutral,Frequent,27.688921
9,Neutral,Infrequent,10.460529


### **BONUS OPTION 1 — Simple Predictive Model**

Goal :
Predict whether a trader will be profitable (1) or not (0) on a given day.

Step 1: Create Target Variable

In [159]:
# Daily PnL per account
daily = merged.groupby(['Account', 'date']).agg({
    'Closed PnL': 'sum',
    'Size USD': 'mean',
    'win': 'mean',
    'classification': 'first'
}).reset_index()

# Target: profitable or not
daily['profitable'] = (daily['Closed PnL'] > 0).astype(int)

daily.head()

,Account,date,Closed PnL,Size USD,win,classification,profitable
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-06-12,-175611.000056,36736.362424,0.000000,Greed,0
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-11,0.000000,5089.718249,0.000000,Extreme Greed,0
2,0x083384f897ee0f19899168e3b1bec365f52a9012,2025-01-03,9482.221441,2985.797556,0.800000,Greed,1
3,0x083384f897ee0f19899168e3b1bec365f52a9012,2025-02-02,76710.000000,185847.000000,0.900000,Greed,1
4,0x083384f897ee0f19899168e3b1bec365f52a9012,2025-03-02,101011.685664,23593.019857,0.305125,Fear,1


Step 2: Encode sentiment

In [160]:
daily['classification'] = daily['classification'].map({'Fear': 0, 'Greed': 1})

Step 3: Features & Model

In [162]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = daily[['Size USD', 'win', 'classification']]
y = daily['profitable']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.97      0.87      0.92        69
           1       0.93      0.98      0.95       113

    accuracy                           0.94       182
   macro avg       0.95      0.93      0.93       182
weighted avg       0.94      0.94      0.94       182



A classification model was developed to predict trader profitability using sentiment and behavioral features. The model demonstrates that sentiment and trading patterns can partially explain profitability.


### **BONUS OPTION 2 — Trader Clustering**

Goal :
Group traders into behavioral archetypes

Step 1: Create features per trader

In [165]:
trader_features = merged.groupby('Account').agg({
    'Closed PnL': 'mean',
    'Size USD': 'mean',
    'win': 'mean'
}).reset_index()

Step 2: Apply K-Means

In [167]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3)
trader_features['cluster'] = kmeans.fit_predict(
    trader_features[['Closed PnL', 'Size USD', 'win']]
)

trader_features.head()

,Account,Closed PnL,Size USD,win,cluster
0,0x083384f897ee0f19899168e3b1bec365f52a9012,419.127768,16159.576734,0.359612,2
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,6.577654,1653.226327,0.442720,0
2,0x271b280974205ca63b716753467d5a371de622ab,-18.492043,8893.000898,0.301917,0
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,9.951530,507.626933,0.438585,0
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,52.071011,3138.894782,0.519914,0


Step 3: Interpret clusters

Traders were segmented into distinct behavioral clusters, revealing different trading styles such as conservative, aggressive, and consistent performers.

In [175]:
trader_features.groupby('cluster')[['Closed PnL', 'Size USD', 'win']].mean()

,Closed PnL,Size USD,win
cluster,,,
0,92.598589,3646.244855,0.404936
1,68.684419,34396.580284,0.401193
2,154.838854,18587.434539,0.386196


### **BONUS OPTION 3 — Streamlit Dashboard**